In [ ]:
import os, torch, cv2, numpy as np
from PIL import Image
from torch.utils.data import Dataset
from pycocotools.coco import COCO
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_root = "/kaggle/input/datasets/sahgopal/aquarium/train"
valid_root = "/kaggle/input/datasets/sahgopal/aquarium/valid"

In [ ]:
class AquariumDataset(Dataset):

    def __init__(self, root):
        self.root = root
        self.coco = COCO(os.path.join(root,"_annotations.coco.json"))
        self.ids = list(self.coco.imgs.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):

        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.root,img_info["file_name"])

        img = Image.open(path).convert("RGB")
        img = torch.tensor(np.array(img)).permute(2,0,1)/255.

        boxes=[]
        labels=[]

        for a in anns:
            x,y,w,h = a["bbox"]
            boxes.append([x,y,x+w,y+h])
            labels.append(a["category_id"])

        return img.float(),{
            "boxes":torch.tensor(boxes),
            "labels":torch.tensor(labels)
        }

In [ ]:
train_dataset = AquariumDataset(train_root)
valid_dataset = AquariumDataset(valid_root)

print("Train:",len(train_dataset))
print("Valid:",len(valid_dataset))

In [ ]:
img,target = train_dataset[0]
print(img.shape,target["boxes"].shape)

In [ ]:
import torchvision
import torch.nn as nn
from torchvision.ops import roi_pool

class FastRCNN(nn.Module):

    def __init__(self,num_classes):

        super().__init__()

        backbone = torchvision.models.resnet18(weights="DEFAULT")
        self.features = nn.Sequential(*list(backbone.children())[:-2])

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*7*7,1024),
            nn.ReLU(),
            nn.Linear(1024,1024),
            nn.ReLU()
        )

        self.cls = nn.Linear(1024,num_classes)
        self.box = nn.Linear(1024,num_classes*4)

    def forward(self,images,proposals):

        feats = self.features(images)

        pooled = roi_pool(
            feats,
            proposals,
            output_size=(7,7),
            spatial_scale=feats.shape[-1]/images.shape[-1]
        )

        x = self.fc(pooled)

        return self.cls(x), self.box(x)

In [ ]:
NUM_CLASSES = len(train_dataset.coco.cats)+1

model = FastRCNN(NUM_CLASSES).to(device)
print("Classes:",NUM_CLASSES)

In [ ]:
def iou(a,b):
    xA=max(a[0],b[0])
    yA=max(a[1],b[1])
    xB=min(a[2],b[2])
    yB=min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    areaA=(a[2]-a[0])*(a[3]-a[1])
    areaB=(b[2]-b[0])*(b[3]-b[1])
    return inter/(areaA+areaB-inter+1e-6)

def get_proposals(img):

    ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    ss.setBaseImage(np.array(img))
    ss.switchToSelectiveSearchFast()

    rects = ss.process()[:300]

    out=[]
    for x,y,w,h in rects:
        out.append([x,y,x+w,y+h])

    return torch.tensor(out,dtype=torch.float32)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.CrossEntropyLoss()

EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for i in range(len(train_dataset)):

        img,target = train_dataset[i]

        gt_boxes = target["boxes"].to(device)
        gt_labels = target["labels"].to(device)

        batch = torch.zeros((gt_boxes.shape[0],1),device=device)
        rois = torch.cat([batch,gt_boxes],1)

        scores,_ = model(img.unsqueeze(0).to(device), rois)

        loss = loss_fn(scores, gt_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = scores.argmax(1)
        train_correct += (preds == gt_labels).sum().item()
        train_total += gt_labels.numel()

    train_loss /= len(train_dataset)
    train_acc = train_correct / train_total

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for i in range(len(valid_dataset)):

            img,target = valid_dataset[i]

            gt_boxes = target["boxes"].to(device)
            gt_labels = target["labels"].to(device)

            batch = torch.zeros((gt_boxes.shape[0],1),device=device)
            rois = torch.cat([batch,gt_boxes],1)

            scores,_ = model(img.unsqueeze(0).to(device), rois)

            loss = loss_fn(scores, gt_labels)

            val_loss += loss.item()

            preds = scores.argmax(1)
            val_correct += (preds == gt_labels).sum().item()
            val_total += gt_labels.numel()

    val_loss /= len(valid_dataset)
    val_acc = val_correct / val_total

    print(f"""
Epoch {epoch+1}
Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}
Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}
""")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

y_true=[]
y_pred=[]

MAX_IMAGES = 20

for i in range(min(MAX_IMAGES, len(train_dataset))):
for i in range(min(MAX_IMAGES, len(valid_dataset))):

    img,target = valid_dataset[i]

    gt_boxes = target["boxes"].to(device)
    gt_labels = target["labels"]

    # build ROI format
    batch = torch.zeros((gt_boxes.shape[0],1),device=device)
    rois = torch.cat([batch,gt_boxes],1)

    with torch.no_grad():
        scores,_ = model(img.unsqueeze(0).to(device), rois)

    pred = scores.argmax(1).cpu()

    y_true.extend(gt_labels.tolist())
    y_pred.extend(pred.tolist())

print(confusion_matrix(y_true,y_pred))
print(classification_report(y_true,y_pred))

In [ ]:
def apply_deltas(box, delta):

    w = box[2] - box[0]
    h = box[3] - box[1]

    cx = box[0] + 0.5 * w
    cy = box[1] + 0.5 * h

    dx, dy, dw, dh = delta

    cx2 = cx + dx * w
    cy2 = cy + dy * h

    w2 = w * torch.exp(dw)
    h2 = h * torch.exp(dh)

    x1 = cx2 - 0.5 * w2
    y1 = cy2 - 0.5 * h2
    x2 = cx2 + 0.5 * w2
    y2 = cy2 + 0.5 * h2

    return torch.stack([x1, y1, x2, y2])

In [ ]:
model.eval()

img,target = valid_dataset[0]
orig = img.permute(1,2,0).numpy()

gt_boxes = target["boxes"]
gt_labels = target["labels"]

# build ROI format
batch = torch.zeros((gt_boxes.shape[0],1),device=device)
rois = torch.cat([batch,gt_boxes.to(device)],1)

with torch.no_grad():
    scores,_ = model(img.unsqueeze(0).to(device), rois)

pred = scores.softmax(1)
cls = pred.argmax(1).cpu()
conf = pred.max(1).values.cpu()

plt.figure(figsize=(8,8))
plt.imshow(orig)
ax = plt.gca()

for i in range(len(gt_boxes)):
    x1,y1,x2,y2 = gt_boxes[i]

    ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,
        fill=False,color='lime',linewidth=2))

    ax.text(x1,y1,f"{cls[i]}:{conf[i]:.2f}",color="yellow")

plt.show()